## No memory

In [ ]:
%run langchain_common.py

# make sure to start mlflow server before running this notebook
# example(run this from terminal): mlflow ui --port 5000 

mlflow.set_experiment("langchain_agents_memory_demo")

agent = create_agent(llm_noreason)

In [ ]:
question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)
pp(response["messages"][-1].content)

In [ ]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pp(response["messages"][-1].content)

## Memory

In [ ]:
# Checkpoints persist conversation state between turns so the agent can remember prior context.
# InMemorySaver is an in-process checkpoint backend: fast and simple for demos, but non-persistent across kernel restarts.

# Clarification:
# `messages` in agent.invoke(...) is only the per-call input payload (what you send this turn).
# Checkpoint memory is different: it stores and reloads the full conversation state across turns.
# That persisted state is keyed by `thread_id`, so memory works only when the same thread_id is reused.
# In short: message context is request-scoped; checkpoint memory is thread-scoped and persistent (in-process here).

from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    llm_noreason,
    checkpointer=InMemorySaver(),  
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [ ]:
pp(response["messages"][-1].content)

In [ ]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

resp = response["messages"][-1].content
pp(resp)

In [ ]:
# Demonstrate storing custom data in memory for the same thread
user_data = {
    "name": "Ali",
    "favorite_color": "green",
    "city": "Lahore",
    "loyalty_tier": "gold"
}

# Save custom data into the thread memory
save_message = HumanMessage(
    content=(
        "Store this custom data in memory and use it for future answers:\n"
        f"{user_data}"
    )
)
agent.invoke({"messages": [save_message]}, config)

# Ask a follow-up question that requires the stored custom data
question = HumanMessage(
    content="From my saved profile, what city do I live in and what is my loyalty tier?"
)
response = agent.invoke({"messages": [question]}, config)
pp(response["messages"][-1].content)

In [ ]:
# Inspect recent messages stored in memory for this thread
state = agent.get_state(config)
pp(state.values["messages"][-4:])

In [ ]:
import uuid

def new_conversation_id() -> str:
    return str(uuid.uuid4())
  
def make_thread_config(user_id: str) -> dict:
    conversation_id = new_conversation_id()
    return {"configurable": {"thread_id": f"user-{user_id}:conv-{conversation_id}"}}

def ask(thread_config, text):
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, thread_config)
    result =  result["messages"][-1].content
    pp(thread_config["configurable"]["thread_id"] + ":" + result)

In [ ]:

# Scenario 1: Thread isolation (different users/sessions)
config_a = make_thread_config("27100001")
config_b = make_thread_config("27100002")

ask(config_a, "My preferred database is PostgreSQL.")
ask(config_b, "My preferred database is MongoDB.")

ask(config_a, 'What database do I prefer?')
ask(config_b, 'What database do I prefer?')

In [ ]:
# Scenario 2: Preference update within the same thread
config_update = make_thread_config("27100003")

ask(config_update, "My deployment region is East US.")
ask(config_update, "Actually, update that: my deployment region is West Europe.")
ask(config_update, 'Which deployment region should you use now?')

In [ ]:

# Scenario 3: Multi-step task memory
config_task = make_thread_config("27100004")

ask(config_task, "I'm building a chatbot MVP. Deadline is Friday. Budget is $2,000.")
pp(f"Scenario 3A: Recall previous constraints:")
ask(config_task, 'Summarize my constraints in one sentence.')
pp(f"Scenario 3B: Suggest next steps:")
ask(config_task, 'Given those constraints, suggest 3 practical next steps.')

task_state = agent.get_state(config_task)
pp("\nRecent messages in Scenario 3 thread:")
pp(task_state.values["messages"][-4:])